# CoWork Enterprise Demo — Cleanup
## Notebook 08 — Cleanup

_Icons used throughout: 🛠️ Action  📌 Note  🔹 Info_

---

### What This Notebook Does:

1. 🛠️ Removes **only** the demo footprint (agents, budget, quota, masking, DB, warehouse, roles)
2. 🛠️ Removes our agents from the shared CoWork object **without dropping the object**
3. 🛠️ Verifies the account is back to its prior state

---

### Why This Notebook Is Deliberately Narrow:

📌 **This notebook must NOT touch** shared state: the CoWork object `SNOWFLAKE_INTELLIGENCE_OBJECT_DEFAULT` (it only removes *our* agents from it), `CORTEX_USER`/`PUBLIC`, `CORTEX_ENABLED_CROSS_REGION`, or account `AI_SETTINGS` — all pre-existed and are shared with other teams.

🔹 It removes both the dev agent (`DEMO_AGENT`) and the promoted agent (`AGENTS_PROD.DEMO_AGENT`). If a notebook that added one of them wasn't run, that DROP simply reports it isn't present — continue to the next cell.

📌 The CoWork interface branding from Notebook 05 is intentionally **left in place**; re-run the `ALTER SNOWFLAKE INTELLIGENCE ... SET BRAND_NAME = ...` with your own values to revert it.

---

### Estimated Time: **5 minutes**

### Prerequisites:
- Run this **last**, after you're done with Notebooks 00–07

## Step 1: Remove Our Agents from the CoWork Object

🛠️ Detach both agents from the shared singleton — the object itself is preserved for other teams.

In [ ]:
%%sql -r remove_from_cowork
USE ROLE ACCOUNTADMIN;
ALTER SNOWFLAKE INTELLIGENCE SNOWFLAKE_INTELLIGENCE_OBJECT_DEFAULT DROP AGENT COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT;


In [ ]:
%%sql -r remove_prod_from_cowork
USE ROLE ACCOUNTADMIN;
ALTER SNOWFLAKE INTELLIGENCE SNOWFLAKE_INTELLIGENCE_OBJECT_DEFAULT DROP AGENT COWORK_ENTERPRISE_DEMO.AGENTS_PROD.DEMO_AGENT;


## Step 2: Reset the CoWork Object Branding

🛠️ The CoWork object (`SNOWFLAKE_INTELLIGENCE_OBJECT_DEFAULT`) is a shared singleton. At go-live (Notebook 05) we set its `BRAND_NAME`, `WELCOME_MESSAGE`, and accent colours. Because branding is **account-global**, cleanup reverts it to Snowflake defaults so no demo branding lingers for other teams.

🔹 `UNSET` resets each property to its system default (the object itself is never dropped).

In [ ]:
%%sql -r reset_branding
-- Reset the shared CoWork object branding to Snowflake defaults (undoes NB05 go-live branding).
USE ROLE ACCOUNTADMIN;
ALTER SNOWFLAKE INTELLIGENCE SNOWFLAKE_INTELLIGENCE_OBJECT_DEFAULT
  UNSET BRAND_NAME, WELCOME_MESSAGE, ACCENT_COLOR_LIGHT, ACCENT_COLOR_DARK;
USE ROLE COWORK_ENTERPRISE_DEMO_ADMIN;
SELECT 'CoWork object branding reset to Snowflake defaults' AS STATUS;

## Step 3: Drop the Demo Objects

🛠️ Drop the budget, quota, and masking policy, then the database (CASCADE), warehouse, and roles. Dropping the database removes the agents, semantic view, search service, eval dataset, and stage in one step.

In [ ]:
%%sql -r drop_budget
DROP SNOWFLAKE.CORE.BUDGET IF EXISTS COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT_BUDGET;
DROP SNOWFLAKE.CORE.BUDGET IF EXISTS COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_TEAM_BUDGET;


In [ ]:
%%sql -r drop_quota
USE ROLE ACCOUNTADMIN;
DROP SNOWFLAKE.CORE.QUOTA IF EXISTS COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT_QUOTA;
USE ROLE COWORK_ENTERPRISE_DEMO_ADMIN;


In [ ]:
%%sql -r unset_masking
ALTER TABLE IF EXISTS COWORK_ENTERPRISE_DEMO.ANALYTICS.CLIENTS MODIFY COLUMN EMAIL UNSET MASKING POLICY;


In [ ]:
%%sql -r drop_objects
USE ROLE ACCOUNTADMIN;
DROP USER IF EXISTS COWORK_ENTERPRISE_DEMO_BIZ_USER;
DROP DATABASE IF EXISTS COWORK_ENTERPRISE_DEMO CASCADE;
DROP WAREHOUSE IF EXISTS COWORK_ENTERPRISE_DEMO_WH;
DROP ROLE IF EXISTS COWORK_ENTERPRISE_DEMO_SI_USER;
DROP ROLE IF EXISTS COWORK_ENTERPRISE_DEMO_ADMIN;


## Step 4: Verify Clean

📌 All should return no rows / gone. The shared CoWork object should still exist (we only removed our agents from it).

In [ ]:
%%sql -r verify_db_gone
SHOW DATABASES LIKE 'COWORK_ENTERPRISE_DEMO';


In [ ]:
%%sql -r verify_role_gone
SHOW ROLES LIKE 'COWORK_ENTERPRISE_DEMO%';


In [ ]:
%%sql -r verify_object_kept
SHOW SNOWFLAKE INTELLIGENCES;


## ✅ Notebook 08 Complete

### What You Just Built:
- A clean account — all `COWORK_ENTERPRISE_DEMO*` objects removed
- The shared CoWork object preserved, with our agents detached and its branding reset to Snowflake defaults

---

### Key Points:
- The demo is fully isolated, so cleanup is a handful of drops plus detaching our agents.
- Shared, pre-existing state (CoWork object, PUBLIC grants, guardrails) is never touched.

---

### Next:

That's the full lifecycle: context → govern → build → cost → evaluate/go-live → dev→prod → continuous improvement → cleanup. See `PRODUCTION_HARDENING.md` for the account-level hardening to run in a real deployment.